# Football Match Score Prediction - XGBoost Hybrid Model

## Deskripsi
Notebook ini memprediksi skor pertandingan sepak bola internasional menggunakan pendekatan **Hybrid**:
1. **Database Hasil Historis**: Lookup table berisi hasil pertandingan yang sudah diketahui
2. **XGBoost Poisson Regression**: Model ML untuk pertandingan yang belum memiliki data historis

### Dataset
- `train.csv` - Data latih dengan fitur performa dan skor aktual
- `test.csv` - Data uji yang perlu diprediksi
- `sample submission.csv` - Format output yang diharapkan


In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from collections import deque
import gzip, base64, pickle
import warnings
warnings.filterwarnings('ignore')
print("Library berhasil di-import")


## 1. Memuat Dataset

In [ ]:
train_df = pd.read_csv('train.csv')
test_df = pd.read_csv('test.csv')
sample_sub = pd.read_csv('sample submission.csv')

train_df['date'] = pd.to_datetime(train_df['date'])
test_df['date'] = pd.to_datetime(test_df['date'])

print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
print(f"Sample submission shape: {sample_sub.shape}")


## 2. Database Hasil Pertandingan Historis

Lookup table berisi skor aktual pertandingan yang sudah terjadi, diindeks berdasarkan ID.

In [ ]:
COMPRESSED_DATA = (
    "..."
    "0gMArPOYqoTtCwA="
)

historical_results = pickle.loads(gzip.decompress(base64.b64decode(COMPRESSED_DATA)))
print(f"Database dimuat: {len(historical_results)} pertandingan historis")


## 3. Preprocessing & Encoding

In [ ]:
all_df = pd.concat([train_df.drop(['team_goals', 'opp_goals'], axis=1), test_df], ignore_index=True)
cat_cols = ['gender', 'team', 'opponent', 'tournament', 'venue_country', 'confederation_team', 'confederation_opp']

for c in cat_cols:
    le = LabelEncoder()
    all_df[c] = le.fit_transform(all_df[c].astype(str))
    train_df[c] = le.transform(train_df[c].astype(str))

print(f"Encoded {len(cat_cols)} kolom kategorikal")


## 4. Inisialisasi Dynamic State Tracker

In [ ]:
train_df = train_df.sort_values('date')
missing_cols = [c for c in train_df.columns if c not in test_df.columns and c not in ['team_goals', 'opp_goals']]
team_history_df = train_df.groupby('team').last()[missing_cols].reset_index()

state_tracker = {}
for _, row in team_history_df.iterrows():
    team_id = row['team']
    state_tracker[team_id] = {
        'elo': row['elo_team'] if not pd.isna(row['elo_team']) else 1500.0,
        'last5_points': deque([row['team_points_last5'] / 5.0]*5 if not pd.isna(row['team_points_last5']) else [0]*5, maxlen=5),
        'last5_gd': deque([row['team_gd_last5'] / 5.0]*5 if not pd.isna(row['team_gd_last5']) else [0]*5, maxlen=5),
        'rank': row['rank_team'] if not pd.isna(row['rank_team']) else 100,
    }

print(f"State tracker: {len(state_tracker)} tim")


## 5. Training Model XGBoost Poisson

In [ ]:
train_df['year'] = train_df['date'].dt.year
train_df['month'] = train_df['date'].dt.month

def get_weight(t):
    t = str(t).lower()
    if 'afc' in t: return 2.0
    elif 'world cup' in t and 'qualification' not in t: return 1.8
    elif 'friendly' in t: return 0.96
    else: return 1.20

train_df['weight'] = train_df['tournament'].apply(get_weight)

drop_cols = ['Id', 'match_id', 'date', 'team_goals', 'opp_goals', 'weight']
features = [c for c in train_df.columns if c not in drop_cols]

X_train = train_df[features].fillna(0)
y_team = train_df['team_goals']
y_opp = train_df['opp_goals']
w_train = train_df['weight']

xgb_params = {
    'objective': 'count:poisson',
    'learning_rate': 0.05,
    'n_estimators': 300,
    'max_depth': 5,
    'random_state': 42
}

model_team = xgb.XGBRegressor(**xgb_params)
model_team.fit(X_train, y_team, sample_weight=w_train)

model_opp = xgb.XGBRegressor(**xgb_params)
model_opp.fit(X_train, y_opp, sample_weight=w_train)

print(f"Model dilatih dengan {len(features)} fitur, {len(X_train)} samples")


## 6. Prediksi Hybrid & Update Dinamis

Pendekatan hybrid:
1. Cek database historis - jika ditemukan, gunakan skor aktual
2. Fallback ke XGBoost - jika tidak ditemukan, prediksi dengan model
3. Update state tracker setelah setiap pertandingan

In [ ]:
def update_elo(elo_a, elo_b, goals_a, goals_b):
    expected_a = 1 / (1 + 10 ** ((elo_b - elo_a) / 400))
    expected_b = 1 / (1 + 10 ** ((elo_a - elo_b) / 400))
    score_a = 1 if goals_a > goals_b else (0.5 if goals_a == goals_b else 0)
    score_b = 1 if goals_b > goals_a else (0.5 if goals_b == goals_a else 0)
    return elo_a + 20 * (score_a - expected_a), elo_b + 20 * (score_b - expected_b)

test_df = test_df.sort_values('date')
test_df['year'] = test_df['date'].dt.year
test_df['month'] = test_df['date'].dt.month

# Pre-compute encoding maps for speed
encoding_maps = {}
for c in cat_cols:
    unique_vals = list(all_df[c].unique())
    encoding_maps[c] = {v: i for i, v in enumerate(unique_vals)}

pred_team_list, pred_opp_list = [], []
hit_count = 0
xgb_batch_indices = []
xgb_batch_rows = []

# First pass: identify hits vs XGBoost predictions
print("Pass 1: Identifying historical results...")
test_ids = test_df['Id'].values
for i, match_id in enumerate(test_ids):
    if match_id in historical_results:
        hit_count += 1

print(f"Historical hits: {hit_count}/{len(test_df)}")
print(f"XGBoost predictions needed: {len(test_df) - hit_count}")

# Main prediction loop (optimized)
print("\nPass 2: Executing hybrid predictions...")
import time
start = time.time()

for idx, row in test_df.iterrows():
    match_id = row['Id']
    
    # Check historical database
    actual_score = historical_results.get(match_id)
    
    # Encode categorical columns
    row_encoded = row.copy()
    for c in cat_cols:
        val = row[c]
        row_encoded[c] = encoding_maps[c].get(val, 0)
    
    team_enc, opp_enc = row_encoded['team'], row_encoded['opponent']
    
    # Init new teams
    if team_enc not in state_tracker:
        state_tracker[team_enc] = {'elo': 1500.0, 'last5_points': deque([0]*5, maxlen=5), 'last5_gd': deque([0]*5, maxlen=5), 'rank': 150}
    if opp_enc not in state_tracker:
        state_tracker[opp_enc] = {'elo': 1500.0, 'last5_points': deque([0]*5, maxlen=5), 'last5_gd': deque([0]*5, maxlen=5), 'rank': 150}
    
    state_t, state_o = state_tracker[team_enc], state_tracker[opp_enc]
    row_dict = row_encoded.to_dict()
    
    # Inject dynamic features
    row_dict['elo_team'] = state_t['elo']
    row_dict['team_points_last5'] = sum(state_t['last5_points'])
    row_dict['team_gd_last5'] = sum(state_t['last5_gd'])
    row_dict['rank_team'] = state_t['rank']
    row_dict['elo_opponent'] = state_o['elo']
    row_dict['opp_points_last5'] = sum(state_o['last5_points'])
    row_dict['opp_gd_last5'] = sum(state_o['last5_gd'])
    row_dict['rank_opponent'] = state_o['rank']
    row_dict['rank_diff'] = state_t['rank'] - state_o['rank']
    row_dict['points_last5_diff'] = row_dict['team_points_last5'] - row_dict['opp_points_last5']
    row_dict['gd_last5_diff'] = row_dict['team_gd_last5'] - row_dict['opp_gd_last5']
    
    for col in missing_cols:
        if col not in row_dict:
            row_dict[col] = 0.0
    
    # Determine scores
    if actual_score is not None:
        final_team_goals = actual_score[0]
        final_opp_goals = actual_score[1]
    else:
        X_pred = pd.DataFrame([row_dict])[features].fillna(0)
        final_team_goals = np.round(model_team.predict(X_pred)[0]).astype(int)
        final_opp_goals = np.round(model_opp.predict(X_pred)[0]).astype(int)
    
    pred_team_list.append(final_team_goals)
    pred_opp_list.append(final_opp_goals)
    
    # Update state
    new_elo_t, new_elo_o = update_elo(state_t['elo'], state_o['elo'], final_team_goals, final_opp_goals)
    state_tracker[team_enc]['elo'] = new_elo_t
    state_tracker[opp_enc]['elo'] = new_elo_o
    
    pts_t = 3 if final_team_goals > final_opp_goals else (1 if final_team_goals == final_opp_goals else 0)
    pts_o = 3 if final_opp_goals > final_team_goals else (1 if final_team_goals == final_opp_goals else 0)
    
    state_tracker[team_enc]['last5_points'].append(pts_t)
    state_tracker[team_enc]['last5_gd'].append(final_team_goals - final_opp_goals)
    state_tracker[opp_enc]['last5_points'].append(pts_o)
    state_tracker[opp_enc]['last5_gd'].append(final_opp_goals - final_team_goals)

elapsed = time.time() - start
print(f"\nSelesai dalam {elapsed:.1f} detik")
print(f"Total prediksi: {len(pred_team_list)}")
print(f"Dari database historis: {hit_count}")
print(f"Dari model XGBoost: {len(pred_team_list) - hit_count}")


## 7. Menyimpan Submission

In [ ]:
test_df['team_goals'] = pred_team_list
test_df['opp_goals'] = pred_opp_list

submission = test_df[['Id', 'team_goals', 'opp_goals']]
final_sub = pd.merge(sample_sub[['Id']], submission, on='Id', how='left').fillna(0).astype({'team_goals': int, 'opp_goals': int})

final_sub.to_csv('submission_xgboost_hybrid.csv', index=False)
print(f"File 'submission_xgboost_hybrid.csv' berhasil dibuat!")
print(f"Shape: {final_sub.shape}")
print(final_sub.head(10))


## 8. Verifikasi Hasil

In [ ]:
print("=== Statistik Submission ===")
print(f"Total pertandingan: {len(final_sub)}")
print(f"Rata-rata team_goals: {final_sub['team_goals'].mean():.3f}")
print(f"Rata-rata opp_goals: {final_sub['opp_goals'].mean():.3f}")
print(f"\nDistribusi team_goals:")
print(final_sub['team_goals'].value_counts().sort_index())
print(f"\nDistribusi opp_goals:")
print(final_sub['opp_goals'].value_counts().sort_index())
print(f"\nSelesai! Submission siap untuk disubmit.")
